# 🏋️ Physio-Vision — Train 5 New Elderly Exercise Models (v2)

This notebook trains **5 new elderly-focused exercise models** using the DeepRehabPile (UI-PRMD) dataset on Google Colab's free GPU.

### v2 Changes (from colleague diagnostic report):
- **Smaller architecture** (64→32) matching the working squat model
- **Lower learning rate** (0.0005) to avoid early plateau
- **More epochs** (100) with patience for longer convergence
- **Smaller batch size** (8) for better gradient signal on small datasets
- **Calf Raise removed from ML training** — uses rule-based scoring (STS mismatch)
- **Diagnostic cells** added for Fold1 evaluation and optimizer step count

| # | Exercise | Dataset | Frames | Output Model |
|---|---|---|---|---|
| 1 | Seated Knee Extension | SASLR | 63 | `knee_extension_robust.keras` |
| 2 | Wall Push-Up | IL | 77 | `wall_pushup_robust.keras` |
| 3 | Seated Hip March | HS | 69 | `hip_march_robust.keras` |
| 4 | Seated W Raise | SSA | 74 | `w_raise_robust.keras` |

> ⚠️ **Calf Raise** is NOT trained here. It uses rule-based scoring in the app because
> the UI-PRMD "STS" dataset (Sit-to-Stand) is biomechanically different from calf raises.

---
⚠️ **Before running:** Go to `Runtime → Change runtime type → T4 GPU`

## 📦 Cell 1 — Setup & GPU Check

In [ ]:
import tensorflow as tf
import numpy as np
import os
import urllib.request

# Verify GPU is available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU detected: {gpus[0].name}")
    print(f"   TensorFlow version: {tf.__version__}")
    print(f"   Keras version: {tf.keras.__version__}")
else:
    print("⚠️  No GPU found! Go to Runtime → Change runtime type → T4 GPU")

# Create output directory
os.makedirs('trained_models', exist_ok=True)
print("\n📁 Output directory: /content/trained_models/")
print("📌 Note the Keras/TF version above — tell your colleague so model loading matches!")

## ⚙️ Cell 2 — Exercise Configuration

4 ML-scored exercises. Calf Raise is excluded (rule-based in app).

In [ ]:
EXERCISES = {
    "knee_extension": {
        "name": "Seated Knee Extension",
        "acronym": "SASLR",          # Standing Active Straight Leg Raise
        "time_steps": 63,
        "num_features": 66,           # 22 joints × 3 dimensions
        "output_model": "knee_extension_robust.keras",
    },
    "wall_pushup": {
        "name": "Wall Push-Up",
        "acronym": "IL",              # Inline Lunge
        "time_steps": 77,
        "num_features": 66,
        "output_model": "wall_pushup_robust.keras",
    },
    # NOTE: calf_raise REMOVED — STS dataset is biomechanically different from calf raises.
    # The app uses rule-based scoring for calf raise instead.
    "hip_march": {
        "name": "Seated Hip March",
        "acronym": "HS",              # Hurdle Step
        "time_steps": 69,
        "num_features": 66,
        "output_model": "hip_march_robust.keras",
    },
    "w_raise": {
        "name": "Seated W Raise (Scapular Retraction)",
        "acronym": "SSA",             # Standing Shoulder Abduction
        "time_steps": 74,
        "num_features": 66,
        "output_model": "w_raise_robust.keras",
    },
}

BASE_URL = "https://maxime-devanne.com/datasets/RehabPile/UIPRMD_reg"

print(f"📋 {len(EXERCISES)} exercises configured (calf_raise excluded — rule-based):")
for key, cfg in EXERCISES.items():
    print(f"   • {cfg['name']} ({cfg['acronym']}, {cfg['time_steps']} frames)")

## 📥 Cell 3 — Download Training Data from DeepRehabPile

In [ ]:
def download_file(url, dest):
    try:
        urllib.request.urlretrieve(url, dest)
        return True, f"{os.path.getsize(dest)/1024:.0f} KB"
    except Exception as e:
        if os.path.exists(dest): os.remove(dest)
        return False, str(e)

folds = [0, 1]
file_templates = ["x_train_fold{}.npy", "y_train_fold{}.npy",
                  "x_test_fold{}.npy",  "y_test_fold{}.npy"]

for key, cfg in EXERCISES.items():
    acr = cfg["acronym"]
    print(f"\n{'='*60}\n  📥 {cfg['name']} ({acr})\n{'='*60}")
    for fold in folds:
        fold_dir = f"UIPRMD_reg/{acr}/fold{fold}"
        os.makedirs(fold_dir, exist_ok=True)
        print(f"\n  fold{fold}/")
        for tmpl in file_templates:
            fname = tmpl.format(fold)
            dest = os.path.join(fold_dir, fname)
            url = f"{BASE_URL}/{acr}/fold{fold}/{fname}"
            if os.path.isfile(dest):
                print(f"    ✓ {fname:<25s} (exists — skipped)")
                continue
            print(f"    ↓ {fname:<25s} ...", end=" ", flush=True)
            ok, info = download_file(url, dest)
            print(f"✓ {info}" if ok else f"✗ {info}")

print(f"\n{'='*60}\n  ✅ All data downloaded!\n{'='*60}")

## 🔬 Cell 4 — Pelvis Anchor Normalization

In [ ]:
def normalize_skeleton(data):
    """
    Pelvis Anchor Normalization (identical to train.py):
      1. Center at Mid-Hip (Joint 0)
      2. Scale so Pelvis Width = 1.0 (Joint 14 <-> Joint 18)
    """
    B, T, F = data.shape
    J = F // 3
    data = data.reshape(B, T, J, 3)
    root = data[:, :, 0:1, :]
    data = data - root
    left_hip = data[:, :, 18:19, :]
    right_hip = data[:, :, 14:15, :]
    pelvis_width = np.linalg.norm(left_hip - right_hip, axis=3, keepdims=True)
    pelvis_width = np.maximum(pelvis_width, 0.0001)
    data = data / pelvis_width
    return data.reshape(B, T, F)

print("✅ Normalization function ready.")

## 🧠 Cell 5 — Model Architecture (v2: Smaller, matching squat model)

**Key change from v1:** Uses `LSTM(64→32)` instead of `LSTM(128→64)`.

The working `deep_squat_robust.keras` uses this smaller architecture.
The UI-PRMD datasets are small (~168 samples across folds) — a smaller
network generalizes better and avoids the early-plateau issue seen in v1.

In [ ]:
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, Bidirectional, Input
from keras.callbacks import ModelCheckpoint, EarlyStopping
from keras.optimizers import Adam

# ========================================================
#  HYPERPARAMETERS — Tuned based on diagnostic findings
# ========================================================
LSTM_1     = 64       # Was 128 in v1 — smaller to match squat model
LSTM_2     = 32       # Was 64 in v1 — smaller to match squat model
DROPOUT    = 0.3
LR         = 0.0005   # Was 0.001 in v1 — lower to avoid early plateau
EPOCHS     = 100      # Was 50 in v1 — more room for convergence
BATCH_SIZE = 8        # Was 16 in v1 — smaller for better gradient signal
# ========================================================

def build_model(time_steps, num_features):
    """Build the Bidirectional LSTM regression model."""
    model = Sequential([
        Input(shape=(time_steps, num_features)),
        Bidirectional(LSTM(LSTM_1, return_sequences=True)),
        Dropout(DROPOUT),
        Bidirectional(LSTM(LSTM_2)),
        Dropout(DROPOUT),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer=Adam(learning_rate=LR), loss='mse', metrics=['mae'])
    return model

print(f"Architecture: BiLSTM({LSTM_1}) → BiLSTM({LSTM_2}) → Dense(32) → Dense(1)")
print(f"LR={LR}, Epochs={EPOCHS}, Batch={BATCH_SIZE}, Dropout={DROPOUT}")

preview = build_model(74, 66)
preview.summary()

## 🚀 Cell 6 — Train ALL Models

Trains each exercise, saves the best model checkpoint, and reports results.

**⏱️ Expected time on T4 GPU: ~10–15 min total (100 epochs × 4 exercises).**

In [ ]:
import time

training_results = {}
training_histories = {}

for key, cfg in EXERCISES.items():
    name       = cfg["name"]
    acr        = cfg["acronym"]
    ts         = cfg["time_steps"]
    nf         = cfg["num_features"]
    model_file = cfg["output_model"]
    data_path  = f"UIPRMD_reg/{acr}/fold0"
    save_path  = f"trained_models/{model_file}"

    print(f"\n{'='*70}")
    print(f"  🏋️ TRAINING: {name}")
    print(f"  📂 Data: {data_path} | Shape: ({ts}, {nf})")
    print(f"  💾 Output: {save_path}")
    print(f"{'='*70}")

    # --- Load data ---
    x_train = np.load(os.path.join(data_path, "x_train_fold0.npy"))
    y_train = np.load(os.path.join(data_path, "y_train_fold0.npy"))
    x_test  = np.load(os.path.join(data_path, "x_test_fold0.npy"))
    y_test  = np.load(os.path.join(data_path, "y_test_fold0.npy"))

    if x_train.shape[1] == nf:
        x_train = x_train.transpose(0, 2, 1)
        x_test  = x_test.transpose(0, 2, 1)

    print(f"  📊 Samples: train={x_train.shape[0]}, test={x_test.shape[0]}")
    print(f"  📊 X_Train shape: {x_train.shape}")

    x_train = normalize_skeleton(x_train)
    x_test  = normalize_skeleton(x_test)

    model = build_model(ts, nf)

    checkpoint = ModelCheckpoint(
        save_path,
        monitor='val_mae',
        save_best_only=True,
        mode='min',
        verbose=1,
    )

    # Early stopping: if val_mae doesn't improve for 25 epochs, stop early
    early_stop = EarlyStopping(
        monitor='val_mae',
        patience=25,
        restore_best_weights=True,
        verbose=1,
    )

    t0 = time.time()
    history = model.fit(
        x_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(x_test, y_test),
        callbacks=[checkpoint, early_stop],
        verbose=1,
    )
    elapsed = time.time() - t0

    best_mae = min(history.history['val_mae'])
    best_epoch = history.history['val_mae'].index(best_mae) + 1
    total_epochs = len(history.history['val_mae'])

    training_results[key] = {
        "name": name,
        "best_val_mae": best_mae,
        "best_epoch": best_epoch,
        "total_epochs": total_epochs,
        "time_seconds": elapsed,
        "model_file": model_file,
    }
    training_histories[key] = history

    print(f"\n  ✅ {name} — Done in {elapsed:.0f}s")
    print(f"     Best MAE: {best_mae:.4f} at epoch {best_epoch}/{total_epochs}")
    if best_epoch < total_epochs * 0.3:
        print(f"     ⚠️  Best checkpoint was EARLY (epoch {best_epoch}) — model may have plateaued.")
        print(f"        Consider: lower LR (0.0003), more epochs (200), or smaller batch (4).")
    elif best_epoch > total_epochs * 0.8:
        print(f"     👍 Best checkpoint was LATE (epoch {best_epoch}) — good sign!")

# ── Summary ──
print(f"\n\n{'='*70}")
print(f"  📊 TRAINING SUMMARY")
print(f"{'='*70}")
print(f"  {'Exercise':<30s}  {'MAE':>8s}  {'Best@':>8s}  {'Time':>8s}")
print(f"  {'─'*30}  {'─'*8}  {'─'*8}  {'─'*8}")
for key, r in training_results.items():
    print(f"  {r['name']:<30s}  {r['best_val_mae']:8.4f}  ep{r['best_epoch']:>3d}    {r['time_seconds']:6.0f}s")
print(f"{'='*70}")

## 📊 Cell 7 — Training Curves (Loss & MAE per Exercise)

In [ ]:
import matplotlib.pyplot as plt

n = len(training_histories)
fig, axes = plt.subplots(n, 2, figsize=(14, 4*n))
if n == 1:
    axes = [axes]

for i, (key, history) in enumerate(training_histories.items()):
    name = EXERCISES[key]['name']
    ax_loss, ax_mae = axes[i]

    ax_loss.plot(history.history['loss'], label='Train', linewidth=1.5)
    ax_loss.plot(history.history['val_loss'], label='Val', linewidth=1.5)
    ax_loss.set_title(f'{name} — Loss (MSE)', fontweight='bold')
    ax_loss.legend()
    ax_loss.grid(True, alpha=0.3)

    ax_mae.plot(history.history['mae'], label='Train', linewidth=1.5)
    ax_mae.plot(history.history['val_mae'], label='Val', linewidth=1.5)
    best = min(history.history['val_mae'])
    best_ep = history.history['val_mae'].index(best)
    ax_mae.axvline(best_ep, color='red', linestyle='--', alpha=0.5, label=f'Best: {best:.4f}')
    ax_mae.set_title(f'{name} — MAE', fontweight='bold')
    ax_mae.legend()
    ax_mae.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Diagnostic: check if val_mae was flat (sign of no learning)
for key, history in training_histories.items():
    vals = history.history['val_mae']
    if max(vals) - min(vals) < 0.01:
        print(f"⚠️  {EXERCISES[key]['name']}: val_mae barely changed ({min(vals):.4f}→{vals[-1]:.4f}) — model may not have learned!")
    else:
        print(f"✅ {EXERCISES[key]['name']}: val_mae range {min(vals):.4f}→{max(vals):.4f} (spread={max(vals)-min(vals):.4f})")

## 🔍 Cell 8 — THE DECISIVE TEST: Evaluate on Unseen Fold1 Data

This is the **ground truth test** from the diagnostic report. It loads data the model
has NEVER seen during training, with real human-rater labels, and checks whether
predictions actually track those labels.

**How to read:**
- MAE < ~0.15 AND predictions track doctor scores → ✅ Model learned
- MAE > ~0.20 AND predictions barely vary → ❌ Model did not learn

In [ ]:
from keras.models import load_model
import random

for key, cfg in EXERCISES.items():
    name       = cfg["name"]
    acr        = cfg["acronym"]
    ts         = cfg["time_steps"]
    nf         = cfg["num_features"]
    model_path = f"trained_models/{cfg['output_model']}"
    fold1_path = f"UIPRMD_reg/{acr}/fold1"

    print(f"\n{'='*70}")
    print(f"  🔍 FOLD1 EVALUATION: {name}")
    print(f"{'='*70}")

    if not os.path.isfile(model_path):
        print(f"  ⚠️  Model not found: {model_path} — skipping")
        continue

    x_exam = np.load(os.path.join(fold1_path, "x_test_fold1.npy"))
    y_exam = np.load(os.path.join(fold1_path, "y_test_fold1.npy"))

    if x_exam.shape[1] == nf:
        x_exam = x_exam.transpose(0, 2, 1)

    x_exam = normalize_skeleton(x_exam)

    model = load_model(model_path)
    loss, mae = model.evaluate(x_exam, y_exam, verbose=0)

    verdict = "✅ GOOD" if mae < 0.15 else "⚠️ MARGINAL" if mae < 0.20 else "❌ POOR"
    print(f"  Fold1 MAE: {mae:.4f} — {verdict}")

    # Blind test: 10 random samples
    print(f"\n  🎯 AI vs Real Doctor (10 random samples):")
    print(f"  {'─'*60}")

    indices = random.sample(range(len(x_exam)), min(10, len(x_exam)))
    predictions = model.predict(x_exam[indices], verbose=0).flatten()

    pred_spread = predictions.max() - predictions.min()
    for i, idx in enumerate(indices):
        doctor = y_exam[idx]
        ai = predictions[i]
        err = abs(doctor - ai)
        emoji = "✅" if err < 0.1 else "⚠️" if err < 0.2 else "❌"
        print(f"  {emoji} Rep #{idx:>3d} | Doctor: {doctor:.3f} | AI: {ai:.3f} | Error: {err:.3f}")

    print(f"  {'─'*60}")
    print(f"  Prediction spread: {pred_spread:.4f}")
    if pred_spread < 0.05:
        print(f"  ⚠️  Predictions barely vary — model is likely outputting near-constant values!")
    else:
        print(f"  ✅ Predictions show meaningful variation.")

## 💾 Cell 9 — Download Trained Models to Your PC

Downloads all `.keras` files. Drop them into your `Physiology-LLM-Capstone/` project root.

In [ ]:
from google.colab import files

print("📁 Trained models:")
print("─" * 50)

model_files = [f for f in os.listdir('trained_models') if f.endswith('.keras')]

for f in sorted(model_files):
    path = os.path.join('trained_models', f)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"  ✅ {f:<35s} ({size_mb:.1f} MB)")

print(f"\n🔽 Downloading {len(model_files)} model(s)...")
print("   Place these in your Physiology-LLM-Capstone/ root folder.\n")

for f in sorted(model_files):
    files.download(os.path.join('trained_models', f))

---

## 🔁 Cell 10 — Retrain a Single Exercise (Experimentation)

Use this cell for **trial-and-error iteration** on one exercise.
Tweak `TARGET`, `EPOCHS`, `BATCH_SIZE`, `LR`, `LSTM_1`, `LSTM_2` to experiment.

In [ ]:
# ════════════════════════════════════════════════════════════
#  🎯 CHANGE THESE TO EXPERIMENT
# ════════════════════════════════════════════════════════════
TARGET          = "knee_extension"  # wall_pushup, hip_march, w_raise
RETRAIN_EPOCHS  = 150
RETRAIN_BATCH   = 4
RETRAIN_LR      = 0.0003
RETRAIN_LSTM_1  = 64
RETRAIN_LSTM_2  = 32
RETRAIN_DROPOUT = 0.25
# ════════════════════════════════════════════════════════════

cfg = EXERCISES[TARGET]
ts, nf = cfg["time_steps"], cfg["num_features"]
data_path = f"UIPRMD_reg/{cfg['acronym']}/fold0"
save_path = f"trained_models/{cfg['output_model']}"

print(f"🎯 Retraining: {cfg['name']}")
print(f"   LSTM=({RETRAIN_LSTM_1},{RETRAIN_LSTM_2}), LR={RETRAIN_LR}, Epochs={RETRAIN_EPOCHS}, Batch={RETRAIN_BATCH}\n")

x_train = np.load(os.path.join(data_path, "x_train_fold0.npy"))
y_train = np.load(os.path.join(data_path, "y_train_fold0.npy"))
x_test  = np.load(os.path.join(data_path, "x_test_fold0.npy"))
y_test  = np.load(os.path.join(data_path, "y_test_fold0.npy"))

if x_train.shape[1] == nf:
    x_train = x_train.transpose(0, 2, 1)
    x_test  = x_test.transpose(0, 2, 1)

x_train = normalize_skeleton(x_train)
x_test  = normalize_skeleton(x_test)

model = Sequential([
    Input(shape=(ts, nf)),
    Bidirectional(LSTM(RETRAIN_LSTM_1, return_sequences=True)),
    Dropout(RETRAIN_DROPOUT),
    Bidirectional(LSTM(RETRAIN_LSTM_2)),
    Dropout(RETRAIN_DROPOUT),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model.compile(optimizer=Adam(learning_rate=RETRAIN_LR), loss='mse', metrics=['mae'])

checkpoint = ModelCheckpoint(save_path, monitor='val_mae', save_best_only=True, mode='min', verbose=1)
early_stop = EarlyStopping(monitor='val_mae', patience=30, restore_best_weights=True, verbose=1)

history = model.fit(
    x_train, y_train,
    epochs=RETRAIN_EPOCHS,
    batch_size=RETRAIN_BATCH,
    validation_data=(x_test, y_test),
    callbacks=[checkpoint, early_stop],
)

best_mae = min(history.history['val_mae'])
best_epoch = history.history['val_mae'].index(best_mae) + 1
print(f"\n✅ Best val_mae: {best_mae:.4f} at epoch {best_epoch} — saved to {save_path}")